# 02: Prepare Benchmarks

**Run once.** Pulls ScienceQA + Mind2Web from HuggingFace, runs the in-repo
preprocessing scripts to drop a unified JSONL + per-task images, and copies
in the synthetic sample fixture. Output lives at
`/content/drive/MyDrive/grounded_vla/data/`.

**Runtime:** any (CPU is fine; no GPU needed). **Cost:** zero compute units.

**Approximate runtime:** ~15 minutes for the configured slice.

In [1]:
# Mount Google Drive. The first run prompts for OAuth; subsequent runs
# in the same session reuse the token automatically.
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
ROOT = pathlib.Path('/content/drive/MyDrive/grounded_vla')
ROOT.mkdir(parents=True, exist_ok=True)
print('drive root:', ROOT)

Mounted at /content/drive
drive root: /content/drive/MyDrive/grounded_vla


In [3]:
# Clone repo into ephemeral /content (fast scratch space).
REPO_URL = 'https://github.com/KarthikRed2000/grounded_vla.git'
import os, subprocess
if not os.path.exists('/content/grounded_vla'):
    subprocess.run(['git', 'clone', REPO_URL, '/content/grounded_vla'], check=True)
%cd /content/grounded_vla

/content/grounded_vla


In [4]:
# Install repo + GPU stack. Quiet to keep the cell output sane.
!pip install -q -e .[gpu,data]

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.5 MB/s eta 0:00:00
  Building editable for grounded_vla (pyproject.toml) ... done


In [5]:
# HF auth (only needed if you increase --limit and start hitting gated splits).
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)
    print('HF auth OK')
except Exception as e:
    print('Skipping HF auth (probably fine):', e)

HF auth OK


In [6]:
import os
DATA = '/content/drive/MyDrive/grounded_vla/data'
os.makedirs(DATA, exist_ok=True)

# ScienceQA (test split, image-bearing rows). 200 tasks ~= 5 minutes.
!python scripts/prepare_scienceqa.py --split test --out-dir {DATA}/scienceqa --limit 200
!ls {DATA}/scienceqa && wc -l {DATA}/scienceqa/test.jsonl

README.md: 10.3kB [00:00, 15.4MB/s]
scienceqa: 425it [00:48,  8.71it/s]
wrote /content/drive/MyDrive/grounded_vla/data/scienceqa/test.jsonl (200 rows)
images	test.jsonl
200 /content/drive/MyDrive/grounded_vla/data/scienceqa/test.jsonl


In [7]:
# Mind2Web (Multimodal variant; the DOM-only `osunlp/Mind2Web` lacks screenshots).
# We probe available splits and fall back gracefully if `test_task` isn't
# exposed in streaming mode. 100 tasks ~= 8-15 min due to screenshot decode.
!python scripts/prepare_mind2web.py \
    --dataset-id osunlp/Multimodal-Mind2Web \
    --split test_task \
    --out-dir {DATA}/mind2web --limit 100
!ls {DATA}/mind2web && ls {DATA}/mind2web/*.jsonl | head && wc -l {DATA}/mind2web/*.jsonl

README.md: 5.55kB [00:00, 10.8MB/s]
Resolving data files: 100% 27/27 [00:00<00:00, 194247.36it/s]
Available splits in osunlp/Multimodal-Mind2Web: ['test_website', 'test_task', 'test_domain', 'train']
Resolving data files: 100% 27/27 [00:00<00:00, 241463.13it/s]
mind2web/test_task: 100it [01:20,  1.24it/s]
wrote /content/drive/MyDrive/grounded_vla/data/mind2web/test_task.jsonl
  rows: 101
  rows_with_image: 100
  images_dir: /content/drive/MyDrive/grounded_vla/data/mind2web/images
images	test_task.jsonl
/content/drive/MyDrive/grounded_vla/data/mind2web/test_task.jsonl
100 /content/drive/MyDrive/grounded_vla/data/mind2web/test_task.jsonl


In [8]:
# Synthetic sample (in-repo fixture; copy into the Drive data folder).
import shutil
from pathlib import Path
src = Path('data/samples')
dst = Path(DATA) / 'synthetic_sample'
dst.mkdir(parents=True, exist_ok=True)
shutil.copytree(src / 'images', dst / 'images', dirs_exist_ok=True)
shutil.copy(src / 'synthetic_sample.jsonl', dst / 'synthetic_sample.jsonl')
print('synthetic sample @', dst)

synthetic sample @ /content/drive/MyDrive/grounded_vla/data/synthetic_sample


In [9]:
import subprocess
subprocess.run(['du', '-sh', DATA])
subprocess.run(['ls', '-R', DATA])

CompletedProcess(args=['ls', '-R', '/content/drive/MyDrive/grounded_vla/data'], returncode=0)

## Done

Data is now on Drive. Move on to `03_run_eval.ipynb`. If you ever want a
larger evaluation slice, increase `--limit` and re-run just the relevant cell.